# DPG 2.0 Current Results

This notebook summarizes the current DPG 2.0 local-explanation results using the merged `_analysis` outputs already generated in this repository.

It covers:
- best next-phase configurations and path-level metrics
- agreement and cohort summaries
- consolidated comparison against legacy DPG, pruning, SHAP, ICE, and LIME
- current exemplar case-study renderings

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 160)

search_roots = [Path.cwd(), Path.cwd().resolve()]
search_roots.extend(Path.cwd().resolve().parents)

repo_root = None
analysis_root = None
for candidate in search_roots:
    candidate_analysis = candidate / 'experiments_local_explanation' / 'experiment_dpg2_next_phase' / '_analysis'
    if candidate_analysis.exists():
        repo_root = candidate
        analysis_root = candidate_analysis
        break

if repo_root is None or analysis_root is None:
    raise FileNotFoundError('Could not locate experiments_local_explanation/experiment_dpg2_next_phase/_analysis from the current working directory.')
baseline_root = repo_root / 'experiments_local_explanation' / 'results_baselines_by_dataset'

priority_dirs = [
    analysis_root / 'comparison_story_phase_c_critical_mix',
    analysis_root / 'comparison_story_phase_c',
    analysis_root / 'comparison_story_local_dpg',
    analysis_root / 'comparison_story_fast',
    analysis_root / 'comparison_story',
    analysis_root / 'comparison_story_horizontal_v2',
]

comparison_dir = None
for candidate in priority_dirs:
    if (candidate / 'consolidated_comparison_summary.csv').exists():
        comparison_dir = candidate
        break

if comparison_dir is None:
    raise FileNotFoundError('No comparison-story directory with consolidated results was found.')

print('analysis_root   :', analysis_root)
print('comparison_dir  :', comparison_dir)
print('baseline_root   :', baseline_root)

In [ ]:
best_configs = pd.read_csv(analysis_root / 'best_configs.csv', low_memory=False)
best_per_sample = pd.read_csv(analysis_root / 'best_per_sample.csv', low_memory=False)
best_cohort_summary = pd.read_csv(analysis_root / 'best_cohort_summary.csv', low_memory=False)
best_agreement_summary = pd.read_csv(analysis_root / 'best_agreement_summary.csv', low_memory=False)

consolidated_summary = pd.read_csv(comparison_dir / 'consolidated_comparison_summary.csv', low_memory=False)
cohort_effect_sizes = pd.read_csv(comparison_dir / 'cohort_effect_sizes.csv', low_memory=False)
case_study_metadata_path = comparison_dir / 'case_study_metadata.csv'
case_study_metadata = pd.read_csv(case_study_metadata_path, low_memory=False) if case_study_metadata_path.exists() else pd.DataFrame()

display(Markdown('## Best Configurations'))
display(best_configs.sort_values(['dataset', 'graph_construction_mode']).reset_index(drop=True))

display(Markdown('## Consolidated Comparison Summary'))
display(consolidated_summary)

display(Markdown('## Best Cohort Summary'))
display(best_cohort_summary.sort_values(['graph_construction_mode', 'cohort_label']).reset_index(drop=True))

display(Markdown('## Best Agreement Summary'))
agreement_label_col = None
for candidate in ['agreement_group', 'disagree_with_model', 'agreement_label']:
    if candidate in best_agreement_summary.columns:
        agreement_label_col = candidate
        break
agreement_sort_cols = ['graph_construction_mode'] + ([agreement_label_col] if agreement_label_col is not None else [])
display(best_agreement_summary.sort_values(agreement_sort_cols).reset_index(drop=True))

In [ ]:
display(Markdown('## Phase C Final Paper Subset'))
final_subset_path = comparison_dir / 'final_paper_subset.csv'
if final_subset_path.exists() and not case_study_metadata.empty:
    final_subset = pd.read_csv(final_subset_path, low_memory=False)
    final_cases = final_subset.merge(
        case_study_metadata,
        on=['case_label', 'dataset', 'sample_idx', 'graph_construction_mode'],
        how='left',
    )
    display(final_cases[[
        'case_label', 'dataset', 'sample_idx', 'graph_construction_mode', 'selection_note',
        'true_label', 'model_pred', 'local_pred', 'critical_node_label',
        'critical_split_depth', 'critical_node_contrast', 'explanation_confidence', 'support_margin'
    ]])
else:
    display(Markdown('No `final_paper_subset.csv` found for the active comparison directory.'))

In [ ]:
metric_cols = [
    'local_matches_model_rate',
    'local_accuracy',
    'avg_evidence_margin_pred_vs_competitor',
    'avg_edge_precision',
    'avg_edge_recall',
    'avg_recombination_rate',
    'avg_explanation_confidence',
]

display(Markdown('## Execution Trace vs Aggregated Transitions'))
mode_summary = best_configs.groupby('graph_construction_mode', as_index=False)[metric_cols].mean()
display(mode_summary)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_specs = [
    ('local_matches_model_rate', 'Fidelity'),
    ('local_accuracy', 'Local Accuracy'),
    ('avg_edge_precision', 'Edge Precision'),
    ('avg_explanation_confidence', 'Explanation Confidence'),
]
for ax, (col, title) in zip(axes.ravel(), plot_specs):
    tmp = best_configs.groupby('graph_construction_mode')[col].mean().reindex(['aggregated_transitions', 'execution_trace'])
    ax.bar(tmp.index, tmp.values, color=['#9fc5e8', '#6aa84f'])
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

display(Markdown('## Key Effect Sizes'))
focus_effects = cohort_effect_sizes[
    cohort_effect_sizes['metric'].isin([
        'explanation_confidence',
        'path_purity',
        'competitor_exposure',
        'edge_precision',
        'recombination_rate',
    ])
].sort_values(['graph_construction_mode', 'comparison', 'metric']).reset_index(drop=True)
display(focus_effects)

In [ ]:
display(Markdown('## Current Per-sample Diagnostics'))
sample_cols = [
    'dataset',
    'graph_construction_mode',
    'sample_idx',
    'cohort_label',
    'disagree_with_model',
    'explanation_confidence',
    'support_margin',
    'node_precision',
    'edge_precision',
    'path_purity',
    'competitor_exposure',
    'recombination_rate',
    'critical_node_label',
    'critical_split_depth',
]
display(best_per_sample[sample_cols].sort_values(['graph_construction_mode', 'dataset', 'sample_idx']).head(20))

display(Markdown('## Consolidated Comparison Ranking'))
ranked = consolidated_summary.sort_values(['mean_fidelity', 'mean_local_accuracy', 'mean_margin'], ascending=False).reset_index(drop=True)
display(ranked)


In [ ]:
display(Markdown('## Case Studies'))
if case_study_metadata.empty:
    display(Markdown(f'No `case_study_metadata.csv` found in `{comparison_dir}`.'))
else:
    display(case_study_metadata)
    for _, row in case_study_metadata.iterrows():
        display(Markdown(
            f"### {row['case_label']} | {row['dataset']} | {row['graph_construction_mode']}  \n"
            f"true={row['true_label']} | model={row['model_pred']} | local={row['local_pred']} | "
            f"conf={row['explanation_confidence']:.3f} | margin={row['support_margin']:.3f} | "
            f"critical={row.get('critical_node_label', 'n/a')} | "
            f"pred_succ={row.get('critical_successor_pred', row.get('critical_successor_pred_label', 'n/a'))} | "
            f"comp_succ={row.get('critical_successor_comp', row.get('critical_successor_comp_label', 'n/a'))}"
        ))
        for key in ['aggregate_png', 'pred_path_png', 'competitor_path_png']:
            path = row.get(key)
            if isinstance(path, str) and Path(path).exists():
                display(Markdown(f'**{key}**'))
                display(Image(filename=str(path)))


In [ ]:
display(Markdown('## Final Paper Figures'))
if final_subset_path.exists() and not case_study_metadata.empty:
    final_subset = pd.read_csv(final_subset_path, low_memory=False)
    final_cases = final_subset.merge(
        case_study_metadata,
        on=['case_label', 'dataset', 'sample_idx', 'graph_construction_mode'],
        how='left',
    )
    for _, row in final_cases.iterrows():
        display(Markdown(
            f"### {row['case_label']} | {row['dataset']} | {row['graph_construction_mode']}  \n"
            f"{row['selection_note']}  \n"
            f"true={row['true_label']} | model={row['model_pred']} | local={row['local_pred']} | "
            f"critical={row.get('critical_node_label', 'n/a')} | depth={row.get('critical_split_depth', 'n/a')} | "
            f"contrast={row.get('critical_node_contrast', 'n/a')}"
        ))
        for key in ['aggregate_png', 'pred_path_png', 'competitor_path_png']:
            path = row.get(key)
            if isinstance(path, str) and Path(path).exists():
                display(Markdown(f'**{key}**'))
                display(Image(filename=str(path)))
else:
    display(Markdown('No final paper subset is available for the active comparison directory.'))
